In [1]:
import ansys.fluent.core as pyfluent
import ansys.fluent.core.parametric as para
from ansys.fluent.visualization import (
    Contour,
    GraphicsWindow,
    Mesh,
    Monitor,
    Pathline,
    Surface,
    Vector,
    XYPlot,
    config,
)
%matplotlib notebook
import numpy as np
import os
import math as math
import time as time
import warnings

In [2]:
working_directory = os.getcwd()

In [3]:
from general_setting_setup import steady_settings
from general_setting_setup import general_settings_transients
from create_graphics import create_particle_track
from peak_velocity_calc import find_peak_velo
from assign_named_expressions import assign_named_expressions

In [4]:
#calculations for inhalation rate and velocity
respiratory_rate = 13.2 #bpm
inhalation_time = 20/respiratory_rate #sec
peak_inhalation_rate = 27*respiratory_rate*np.pi/40 #L/min

In [5]:
seconds_to_simulate = 0.0002
size_of_timestep = 0.0001
number_of_timesteps = math.ceil(seconds_to_simulate/size_of_timestep)
iterations_per_timestep = 50
path_to_mesh_files = "Meshes\\"
list_of_mesh_file_names = ["new-model-only-decimated-1-06skew-032orth-935Mcells.msh.h5",
                           "new-model-only-decimated-1-06skew-036orth-15-Mcells.msh.h5"]

In [ ]:
#launch fluent in solver
solver = pyfluent.launch_fluent(precision='double', 
                                processor_count=12, 
                                ui_mode='gui', 
                                mode='solver', 
                                start_transcript=False)
warnings.filterwarnings("ignore")

In [7]:
#calculate peak inhalation velocity
solver.settings.file.read_mesh(file_name=str(path_to_mesh_files+"test-new-model-only-decimated-1-07skew-025orth-903kcells.msh.h5"))
peak_inhalation_velocity = find_peak_velo(solver_session = solver,
                                          working_directory=working_directory,
                                          peak_flow_rate = peak_inhalation_rate,
                                          calc_area = "nostrils",
                                          area_file_name = "surface_integral",
                                          unit = "L/min")

In [8]:
i=1
for mesh in list_of_mesh_file_names:

# rename previous dpm files
    time.sleep(2)
    try:
        os.rename("interior--facetten.dpm", "interior--facetten_"+name+".dpm")
        os.rename("new-model-only-decimated-1-facetten.dpm", "new-model-only-decimated-1-facetten_" + name + ".dpm")
        os.rename("nostrils.dpm", "nostrils_"+name+".dpm")
        os.rename("oral.dpm", "oral_" + name + ".dpm")
        os.rename("outlet.dpm", "outlet_" + name + ".dpm")
    except:
        print("No dpm files found")
        
#create file name part
    name = mesh[mesh.find("orth")+5:mesh.find(".")]

# read case file for steady
    solver.settings.file.read_case(file_name = "steady_case.cas.h5")

#read mesh file
    solver.settings.file.replace_mesh(file_name=str(path_to_mesh_files+mesh))

# initialization
    initialization = solver.settings.solution.initialization
    initialization.initialization_type = "hybrid"
    initialization.initialize()


#steady state calculation settings and run calculation
    calc_para = solver.settings.solution.run_calculation.parameters
    calc_para.iter_count = 1000
    calc_para.reporting_interval = 5
    solver.settings.solution.run_calculation.calculate()

# saving case and data of steady state simulation step
    path_for_case_data = str("D:\\Lii\\Results\\New-tests\\independence-test\\" + name + "_steady")
    solver.tui.file.write_case_data(path_for_case_data)

# STARTING TRANSIENT SIMULATION FOR MESH INDEPENDENCE
#General settings
    general_settings_transients(solver_session=solver)
    solver.settings.solution.methods.p_v_coupling.flow_scheme = "SIMPLE"

#Create injections
    injections = solver.settings.setup.models.discrete_phase.injections
    injections.create("test-part-01-um")
    inj = injections["test-part-01-um"]
    inj.injection_type.option = "surface"
    inj.initial_values.times(start_time = 0, stop_time = 1.2)
    inj.initial_values.location.injection_surfaces = "nostrils"
    inj.initial_values.velocity.use_face_normal_direction = True
    inj.initial_values.particle_size.option = "uniform"
    inj.initial_values.particle_size.diameter = 0.0000001


    injections.create("test-part-5-um")
    inj = injections["test-part-5-um"]
    inj.injection_type.option = "surface"
    inj.initial_values.times(start_time = 0, stop_time = 1.2)
    inj.initial_values.location.injection_surfaces = "nostrils"
    inj.initial_values.velocity.use_face_normal_direction = True
    inj.initial_values.particle_size.option = "uniform"
    inj.initial_values.particle_size.diameter = 0.000005

#Turn on particle injection, make it unsteady tracking and turn on trap on walls
    solver.settings.setup.models.discrete_phase.general_settings.unsteady_tracking.enabled = True
    solver.settings.setup.models.discrete_phase.physical_models.erosion_accretion_enabled = True
    solver.settings.setup.models.discrete_phase.tracking.max_num_steps = 5000
    solver.settings.setup.boundary_conditions.wall["new-model-only-decimated-1-facetten"].discrete_phase.bc_type = "trap"

#create a named expression for inhalation
    assign_named_expressions(solver_session=solver,
                             peak_velocity = peak_inhalation_velocity,
                             peak_velo_exp_name = "ampl", 
                             breathing_frequency = 0.22, 
                             breathing_freq_exp_name = "freq", 
                             breathing_exp_name = "breathing")

#assign named expression to nostril boundary condition
    solver.settings.setup.boundary_conditions.velocity_inlet["nostrils"].momentum.velocity_magnitude = "breathing"

#start dpm sample
    solver.settings.file.read_journal(file_name_list=["start-transient-dpm-sample.jou"])

# set calculation settings for transient part and run the calculation
    controls = solver.settings.solution.run_calculation.transient_controls
    controls.time_step_count = number_of_timesteps
    controls.time_step_size = size_of_timestep
    controls.max_iter_per_time_step = iterations_per_timestep
    solver.settings.solution.run_calculation.calculate()

#end dpm sample and rename the files
    time.sleep(3)
    solver.settings.file.read_journal(file_name_list=["stop-transient-dpm-sample.jou"])

    case_data_name = str("D:\\Lii\\Results\\New-tests\\independence-test\\" + str(name) + "_test_" + str(i))
    solver.tui.file.write_case_data(case_data_name)
    i += 1


No dpm files found


KeyboardInterrupt: 

In [ ]:
solver.exit()